# Decision tree

Seed 42, hyperparameters from a Bayesian search and features kept only where
out-of-fold permutation importance comes out above zero -- neither chosen by
hand. Fitted and scored below, then pointed at 2026, a season nobody has played
yet.

Every model gets the same three cells: this heading, the fit, and the 2026
prediction. The harness the later ones reuse -- the folds, the search, the
permutation table, the forecast -- is written in this first block and takes the
estimator as an argument, so a second model is a class and a search space.

In [5]:
import warnings

import numpy as np
import optuna
import pandas as pd
from joblib import Parallel, delayed
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from nfl_trees import metrics as metrics_mod
from nfl_trees.config import FeatureConfig
from nfl_trees.data import load_scores
from nfl_trees.features import build_dataset, make_preprocessor

SEED = 42
N_TRIALS = 80
HOLDOUT = 2025
METRICS = ["roc_auc", "accuracy", "log_loss", "brier"]

# The ten candidate features of notebook `02`, by the builder that makes them.
CANDIDATES = {
    "calendar": ["month", "week", "day", "playoff"],
    "win_rates": ["pct_home_win", "pct_away_win"],
    "drive_rates": [
        "home_pct_score_drive",
        "home_pct_allowed_drive",
        "away_pct_score_drive",
        "away_pct_allowed_drive",
    ],
}
CATEGORICAL = {"day"}
ALL_COLUMNS = [c for cols in CANDIDATES.values() for c in cols]


def config_for(columns):
    """A `FeatureConfig` over `columns`, dropping builders nothing survives from."""
    columns = [c for c in ALL_COLUMNS if c in set(columns)]
    return FeatureConfig(
        numeric=[c for c in columns if c not in CATEGORICAL],
        categorical=[c for c in columns if c in CATEGORICAL],
        builders=[b for b, cols in CANDIDATES.items() if set(cols) & set(columns)],
    )


# `drive_rates` folds every plays file, so the first run takes a few seconds.
FULL = config_for(ALL_COLUMNS)
games = load_scores()
X, y, meta = build_dataset(games, FULL, "home_win")
y = y.astype(int)
season = meta["Season"].astype(int)

# 2025 is walled off: neither the search nor the feature selection may see it.
TUNING_SEASONS = [s for s in sorted(season.unique()) if 2013 <= s < HOLDOUT]


# --------------------------------------------------------------------------- #
# the harness, shared by every model in this notebook
# --------------------------------------------------------------------------- #
def fit_model(estimator, train, params, features):
    """The pipeline a config would produce: repo preprocessor, then the model."""
    pipe = Pipeline(
        [
            ("prep", make_preprocessor(features)),
            ("model", estimator(random_state=SEED, **params)),
        ]
    )
    return pipe.fit(X.loc[train, features.columns], y[train])


def over_folds(work, backend):
    """Run `work(season)` once per tuning season, in parallel.

    Threads for a tree: it fits in milliseconds, so shipping a fold to another
    process costs more than the fold. Processes for a forest, which fits in
    seconds and holds the GIL while it does. The model's own `n_jobs` stays at 1
    either way -- two levels of parallelism over twelve cores only take turns.
    """
    return Parallel(n_jobs=-1, prefer=backend)(delayed(work)(s) for s in TUNING_SEASONS)


def cv_auc(estimator, params, features, backend="threads"):
    """Rolling origin: each season scored by a model trained only on earlier ones."""

    def score(s):
        model = fit_model(estimator, (season >= 2011) & (season < s), params, features)
        test = season == s
        return metrics_mod.compute(
            "classification",
            ["roc_auc"],
            y[test].to_numpy(),
            model.predict(X.loc[test, features.columns]),
            model.predict_proba(X.loc[test, features.columns])[:, 1],
        )["roc_auc"]

    return float(np.mean(over_folds(score, backend)))


def tune(estimator, space, features, *, start_from=None, backend="threads", n_trials=N_TRIALS):
    """Bayesian hyperparameter search over `space`, scored by `cv_auc`.

    Optuna's TPE is the optimizer: it fits a density over the parameters that
    scored well and another over the ones that did not, then samples where the
    ratio -- the expected improvement -- is highest. Every trial is drawn from
    what the previous ones showed, which is what separates it from a grid.
    `multivariate=True` fits that model over the knobs jointly rather than one
    at a time, so it can read depth against leaf size instead of guessing each
    alone. `start_from` seeds a known-good point, so a retune cannot come out
    worse than the search it is meant to improve on.
    """

    def objective(trial):
        return cv_auc(estimator, space(trial), features, backend)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
        )
        if start_from is not None:
            study.enqueue_trial(start_from)
        study.optimize(objective, n_trials=n_trials)
    return study.best_params, study.best_value, study


def permutation_table(estimator, params, features, n_repeats=10, backend="threads"):
    """What each column is worth: the roc_auc the model loses when it is shuffled.

    Measured on the held-out season of every fold, never on the training rows,
    where a model can look like it depends on a column it only memorised.
    """

    def shuffle(s):
        model = fit_model(estimator, (season >= 2011) & (season < s), params, features)
        test = season == s
        result = permutation_importance(
            model,
            X.loc[test, features.columns],
            y[test],
            scoring="roc_auc",
            n_repeats=n_repeats,
            random_state=SEED,
        )
        return pd.Series(result.importances_mean, index=features.columns)

    folds = pd.DataFrame(over_folds(shuffle, backend), index=TUNING_SEASONS)
    table = pd.DataFrame({"mean": folds.mean(), "seasons_up": (folds > 0).sum()})
    return table.sort_values("mean", ascending=False)


def selected(importance):
    """Keep the columns whose permutation importance is above zero, drop the rest.

    Above zero means shuffling the column cost the model roc_auc across the
    twelve folds averaged, so it was carrying something. A column at zero was
    never used, and one below zero was noise the model was better off without.
    """
    survives = importance["mean"] > 0
    return list(importance.index[survives]), list(importance.index[~survives])


def holdout_report(estimator, runs):
    """Fit each `(label, params, features, cv)` on 2011-2024 and score it on the holdout."""
    train, test = season.between(2011, 2024), season.eq(HOLDOUT)
    rows = {}
    for label, params, features, cv in runs:
        model = fit_model(estimator, train, params, features)
        rows[label] = {
            "cv_roc_auc": round(cv, 4),
            **{
                k: round(v, 3)
                for k, v in metrics_mod.compute(
                    "classification",
                    METRICS,
                    y[test].to_numpy(),
                    model.predict(X.loc[test, features.columns]),
                    model.predict_proba(X.loc[test, features.columns])[:, 1],
                ).items()
            },
        }
    return pd.DataFrame(rows).T


# --------------------------------------------------------------------------- #
# the model this block is about
# --------------------------------------------------------------------------- #
def tree_space(trial):
    """Five knobs. `min_samples_leaf` starts at 10 because a leaf holding a handful
    of games returns 0 or 1, and `log_loss` punishes every one of those that misses."""
    return dict(
        criterion=trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        max_depth=trial.suggest_int("max_depth", 2, 12),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 250, log=True),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 200, log=True),
        ccp_alpha=trial.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
    )


# -- 1. tune on every candidate feature -------------------------------------- #
params_full, cv_full, _ = tune(DecisionTreeClassifier, tree_space, FULL)
print(f"bayesian search (TPE), {N_TRIALS} trials over {len(TUNING_SEASONS)} folds (2013-2024)")
print("params  : " + "  ".join(f"{k}={v}" for k, v in params_full.items()))
print(f"cv auc  : {cv_full:.4f}   on all {len(FULL.columns)} features")

# -- 2. ask each feature what it is worth ------------------------------------ #
tree_importance = permutation_table(DecisionTreeClassifier, params_full, FULL)
print("\npermutation importance, out of fold: roc_auc lost when the column is shuffled")
print(tree_importance.round(5).to_string())

TREE_KEEP, TREE_DROPPED = selected(tree_importance)
print(f"\nkept    : {', '.join(TREE_KEEP)}")
print(f"dropped : {', '.join(TREE_DROPPED)}")

# -- 3. retune on the survivors ---------------------------------------------- #
TREE_FEATURES = config_for(TREE_KEEP)
TREE_PARAMS, cv_reduced, _ = tune(
    DecisionTreeClassifier, tree_space, TREE_FEATURES, start_from=params_full
)
print(f"\nretuned on the {len(TREE_KEEP)} survivors, builders now {TREE_FEATURES.builders}")
print("params  : " + "  ".join(f"{k}={v}" for k, v in TREE_PARAMS.items()))
print(f"cv auc  : {cv_reduced:.4f}   ({cv_reduced - cv_full:+.4f} against all {len(FULL.columns)})")

# -- 4. the holdout neither step was allowed to see -------------------------- #
train, test = season.between(2011, 2024), season.eq(HOLDOUT)
tree_report = holdout_report(
    DecisionTreeClassifier,
    [
        (f"all {len(FULL.columns)}", params_full, FULL, cv_full),
        (f"kept {len(TREE_KEEP)}", TREE_PARAMS, TREE_FEATURES, cv_reduced),
    ],
)
print(f"\ntrain 2011-2024 ({int(train.sum())} games), holdout {HOLDOUT} ({int(test.sum())} games)")
print(tree_report.to_string())
print(f"always-home : accuracy {y[test].mean():.3f}   <- the bar to clear")

# What the next cell predicts with: the kept features, refit on every game played.
tree = fit_model(DecisionTreeClassifier, season.between(2011, HOLDOUT), TREE_PARAMS, TREE_FEATURES)
print(f"\nrefit on 2011-{HOLDOUT} ({int(season.between(2011, HOLDOUT).sum())} games) to predict 2026")

bayesian search (TPE), 80 trials over 12 folds (2013-2024)
params  : criterion=log_loss  max_depth=5  min_samples_leaf=98  min_samples_split=37  ccp_alpha=2.235791197815289e-06
cv auc  : 0.6148   on all 10 features

permutation importance, out of fold: roc_auc lost when the column is shuffled
                           mean  seasons_up
away_pct_score_drive    0.05277          12
home_pct_allowed_drive  0.02285           9
home_pct_score_drive    0.02034          10
pct_home_win            0.01026           7
away_pct_allowed_drive  0.00757           9
pct_away_win            0.00236           3
week                    0.00013           1
month                   0.00000           0
playoff                 0.00000           0
day                     0.00000           0

kept    : away_pct_score_drive, home_pct_allowed_drive, home_pct_score_drive, pct_home_win, away_pct_allowed_drive, pct_away_win, week
dropped : month, playoff, day

retuned on the 7 survivors, builders now ['calendar', '

In [6]:
from nfl_trees.data import canonical_team
from nfl_trees.features import apply_builders

DIVISIONS = {
    "AFC East": ["BUF", "MIA", "NE", "NYJ"],
    "AFC North": ["BAL", "CIN", "CLE", "PIT"],
    "AFC South": ["HOU", "IND", "JAX", "TEN"],
    "AFC West": ["DEN", "KC", "LAC", "LV"],
    "NFC East": ["DAL", "NYG", "PHI", "WAS"],
    "NFC North": ["CHI", "DET", "GB", "MIN"],
    "NFC South": ["ATL", "CAR", "NO", "TB"],
    "NFC West": ["ARI", "LAR", "SEA", "SF"],
}
DIVISION = {team: div for div, teams in DIVISIONS.items() for team in teams}
TEAMS = sorted(DIVISION)

ROUNDS = {
    "wild_card": ("WILD CARD WEEKEND", "January 10th"),
    "divisional": ("DIVISIONAL PLAYOFFS", "January 17th"),
    "championship": ("CONFERENCE CHAMPIONSHIPS", "January 24th"),
    "super_bowl": ("SUPER BOWL", "February 7th"),
}


def home_win_proba(model, features):
    """Build `frame -> P(the home team wins)`, one probability per row.

    `build_dataset` cannot be used here: it drops every row with a missing
    target, which is all of them when the games have not been played. The
    builders still work -- for any 2026 week the history rates read 2025 in
    full, which is what a forecast made today has to go on.
    """

    def predict(frame):
        design = apply_builders(frame, features.builders)[features.columns]
        numeric = design[features.numeric].apply(pd.to_numeric, errors="coerce").astype(float)
        categorical = design[features.categorical].astype(object).where(
            design[features.categorical].notna(), np.nan
        )
        design = pd.concat([numeric, categorical], axis=1)[features.columns]
        return model.predict_proba(design)[:, 1]

    return predict


def regular_season(predict):
    """Pick all 272 games of 2026 and print the standings they add up to."""
    schedule = load_scores([2026], statuses=("TBD",), include_postseason=False)
    schedule = schedule.assign(
        home=canonical_team(schedule["HomeTeam"]), away=canonical_team(schedule["AwayTeam"])
    )
    schedule["p_home"] = predict(schedule)
    schedule["winner"] = np.where(schedule["p_home"] >= 0.5, schedule["home"], schedule["away"])

    # Two readings of the same 272 probabilities. `W-L` is the record the picks
    # add up to, and a team favoured every week goes 17-0 in it; `exp` sums the
    # probabilities instead, which is the win total the model would bet on.
    wins = schedule["winner"].value_counts().reindex(TEAMS).fillna(0)
    played = (
        schedule["home"].value_counts()
        .add(schedule["away"].value_counts(), fill_value=0)
        .reindex(TEAMS)
    )
    expected = (
        schedule.groupby("home")["p_home"].sum()
        .add(schedule.assign(p=1 - schedule["p_home"]).groupby("away")["p"].sum(), fill_value=0)
        .reindex(TEAMS)
    )

    standings = pd.DataFrame(
        {
            "division": [DIVISION[team] for team in TEAMS],
            "W": wins.astype(int),
            "L": (played - wins).astype(int),
            "exp": expected.round(1),
        },
        index=TEAMS,
    ).sort_values(["W", "exp"], ascending=False)
    standings["record"] = standings["W"].astype(str) + "-" + standings["L"].astype(str)

    print(f"2026 regular season, {len(schedule)} games predicted")
    print("W-L is the record the picks add up to, exp is the summed probabilities\n")
    for division in DIVISIONS:
        block = standings[standings["division"] == division]
        print(
            f"{division:<10}  "
            + "   ".join(f"{t:<3} {r.record:>5} ({r.exp:>4.1f})" for t, r in block.iterrows())
        )
    return standings


def seeds_of(standings, conference):
    """Seeds 1-7: the four division winners by record, then the three best left."""
    table = standings[standings["division"].str.startswith(conference)]
    champions = [block.index[0] for _, block in table.groupby("division", sort=False)]
    won_division = table.index.isin(champions)
    ranked = list(table.index[won_division]) + list(table.index[~won_division][:3])
    return list(enumerate(ranked, start=1))


def round_frame(pairs, round_key):
    week, date = ROUNDS[round_key]
    return pd.DataFrame(
        [
            {
                "Season": 2026, "Week": week, "GameStatus": "TBD", "GameSlot": "Sunday",
                "GameDate": date, "AwayTeam": away, "AwayScore": np.nan,
                "HomeTeam": home, "HomeScore": np.nan,
            }
            for (_, home), (_, away) in pairs
        ]
    )


def play_round(predict, standings, pairs, round_key, label, neutral=False):
    """Run one round. `pairs` is (host, visitor) as (seed, team); returns the winners.

    On a neutral field nobody hosts, and the model has no way to be told that,
    so the Super Bowl is predicted twice -- once with each team as the home side
    -- and the two averaged.
    """
    p_first = predict(round_frame(pairs, round_key))
    if neutral:
        flipped = predict(round_frame([(b, a) for a, b in pairs], round_key))
        p_first = (p_first + (1 - flipped)) / 2

    print(f"\n{label}")
    winners = []
    for (first, second), p in zip(pairs, p_first):
        if abs(p - 0.5) < 1e-9:
            # Both sides landed in the same leaf and the model has nothing left
            # to say, so the pair is decided on the regular season it just
            # predicted rather than on the order the pair happens to be in.
            winner = max((first, second), key=lambda seed: standings.loc[seed[1], "exp"])
            note = "   (tied: settled on expected wins)"
        else:
            winner, note = (first if p > 0.5 else second), ""
        prob = p if winner == first else 1 - p
        print(
            f"  ({second[0]}) {second[1]:<3} {'vs' if neutral else 'at'} ({first[0]}) {first[1]:<3}"
            f"   ->  {winner[1]:<3} {prob:6.1%}{note}"
        )
        winners.append(winner)
    return winners


def playoffs(predict, standings):
    """Seed both conferences off the predicted standings and play the bracket out."""
    print("\n\n2026 playoffs, predicted -- the better seed hosts every round")
    finalists = {}
    for conference in ("AFC", "NFC"):
        seeds = seeds_of(standings, conference)
        print("\n" + conference + " seeds: " + "  ".join(f"{n}.{team}" for n, team in seeds))

        # Seed 1 sits out the wild card round; every later round re-seeds, so the
        # best team left always hosts the worst one left.
        alive = sorted(
            [seeds[0]]
            + play_round(
                predict,
                standings,
                [(seeds[1], seeds[6]), (seeds[2], seeds[5]), (seeds[3], seeds[4])],
                "wild_card",
                f"{conference} wild card   ({seeds[0][1]} on a bye)",
            )
        )
        alive = sorted(
            play_round(
                predict,
                standings,
                [(alive[0], alive[3]), (alive[1], alive[2])],
                "divisional",
                f"{conference} divisional",
            )
        )
        finalists[conference] = play_round(
            predict, standings, [(alive[0], alive[1])], "championship", f"{conference} championship"
        )[0]

    champion = play_round(
        predict,
        standings,
        [(finalists["AFC"], finalists["NFC"])],
        "super_bowl",
        "Super Bowl   (neutral field: predicted from both sides and averaged)",
        neutral=True,
    )[0]
    print(f"\n  champion: {champion[1]}")
    return champion


def forecast_2026(model, features):
    """The whole 2026 season as this model sees it: records, bracket, champion."""
    predict = home_win_proba(model, features)
    standings = regular_season(predict)
    playoffs(predict, standings)
    return standings


tree_standings = forecast_2026(tree, TREE_FEATURES)

2026 regular season, 272 games predicted
W-L is the record the picks add up to, exp is the summed probabilities

AFC East    NE   15-2 (11.0)   BUF   8-9 ( 8.2)   MIA  3-14 ( 7.8)   NYJ  1-16 ( 5.6)
AFC North   CIN  12-5 ( 9.5)   BAL  10-7 ( 9.3)   PIT  5-12 ( 8.2)   CLE  4-13 ( 5.6)
AFC South   HOU  16-1 (11.0)   JAX  15-2 (10.8)   IND  10-7 ( 8.8)   TEN  1-16 ( 4.6)
AFC West    DEN  14-3 (10.2)   KC   12-5 ( 9.9)   LAC  11-6 ( 9.6)   LV   1-16 ( 4.7)
NFC East    PHI  11-6 ( 8.6)   DAL   9-8 ( 9.0)   NYG   8-9 ( 8.7)   WAS  2-15 ( 7.5)
NFC North   DET  12-5 ( 9.5)   MIN  11-6 ( 8.8)   CHI  10-7 ( 8.8)   GB   10-7 ( 8.8)
NFC South   TB    9-8 ( 8.8)   ATL  7-10 ( 8.4)   NO   5-12 ( 7.5)   CAR  1-16 ( 7.1)
NFC West    SEA  15-2 (10.5)   LAR  14-3 (10.2)   SF    8-9 ( 8.6)   ARI  2-15 ( 6.5)


2026 playoffs, predicted -- the better seed hosts every round

AFC seeds: 1.HOU  2.NE  3.DEN  4.CIN  5.JAX  6.KC  7.LAC

AFC wild card   (HOU on a bye)
  (7) LAC at (2) NE    ->  NE   63.5%
  (6) K

# Random forest

The same protocol, the estimator swapped: three hundred trees instead of one,
each grown on its own bootstrap sample of the seasons and offered only a random
subset of the columns at every split, and the answer is their average.

Bagging is the family's first answer to the single tree's defining problem --
change a handful of games and the root split moves, taking every branch with
it. What this block measures is how much that averaging is worth on four
thousand games, printed next to the tree's numbers on the same 2025 holdout.

In [7]:
from functools import partial

from sklearn.ensemble import RandomForestClassifier

# `n_estimators` is not in the search space: adding trees to a forest cannot
# overfit it, so it is a compute setting rather than a regularizer. Measured on
# these folds, cv auc plateaus at 300 (0.6319, against 0.6320 for 500 at 70%
# more time). `n_jobs=1` because the parallelism is spent on the twelve folds.
FOREST = partial(RandomForestClassifier, n_estimators=300, n_jobs=1)

# Forty trials, where the tree got eighty: a trial here costs seconds instead of
# milliseconds, and bagging is the family least sensitive to its knobs.
FOREST_TRIALS = 40


def forest_space(trial):
    """The tree's four knobs that still mean something, plus the two bagging brings.

    `min_samples_leaf` starts at 1 here: a leaf holding three games still returns
    0 or 1, but three hundred of those are averaged before anyone sees the
    number. `max_features` is why bagging works at all -- each split only gets a
    random subset of the columns, so the trees cannot all open with the same
    question -- and `max_samples` is how large each bootstrap sample is.
    `ccp_alpha` is gone: pruning every tree back regularizes the variance the
    averaging is already there to kill.
    """
    return dict(
        criterion=trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        max_depth=trial.suggest_int("max_depth", 2, 24),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 250, log=True),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 200, log=True),
        max_features=trial.suggest_float("max_features", 0.1, 1.0),
        max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
    )


# -- 1. tune on every candidate feature -------------------------------------- #
params_full, cv_full, _ = tune(
    FOREST, forest_space, FULL, backend="processes", n_trials=FOREST_TRIALS
)
print(f"bayesian search (TPE), {FOREST_TRIALS} trials over {len(TUNING_SEASONS)} folds (2013-2024)")
print("params  : " + "  ".join(f"{k}={v}" for k, v in params_full.items()))
print(f"cv auc  : {cv_full:.4f}   on all {len(FULL.columns)} features")

# -- 2. ask each feature what it is worth ------------------------------------ #
forest_importance = permutation_table(FOREST, params_full, FULL, backend="processes")
print("\npermutation importance, out of fold: roc_auc lost when the column is shuffled")
print(forest_importance.round(5).to_string())

FOREST_KEEP, FOREST_DROPPED = selected(forest_importance)
print(f"\nkept    : {', '.join(FOREST_KEEP)}")
print(f"dropped : {', '.join(FOREST_DROPPED)}")

# -- 3. retune on the survivors ---------------------------------------------- #
FOREST_FEATURES = config_for(FOREST_KEEP)
FOREST_PARAMS, cv_reduced, _ = tune(
    FOREST,
    forest_space,
    FOREST_FEATURES,
    start_from=params_full,
    backend="processes",
    n_trials=FOREST_TRIALS,
)
print(f"\nretuned on the {len(FOREST_KEEP)} survivors, builders now {FOREST_FEATURES.builders}")
print("params  : " + "  ".join(f"{k}={v}" for k, v in FOREST_PARAMS.items()))
print(f"cv auc  : {cv_reduced:.4f}   ({cv_reduced - cv_full:+.4f} against all {len(FULL.columns)})")

# -- 4. the holdout neither step was allowed to see -------------------------- #
forest_report = holdout_report(
    FOREST,
    [
        (f"all {len(FULL.columns)}", params_full, FULL, cv_full),
        (f"kept {len(FOREST_KEEP)}", FOREST_PARAMS, FOREST_FEATURES, cv_reduced),
    ],
)
print(f"\ntrain 2011-2024 ({int(train.sum())} games), holdout {HOLDOUT} ({int(test.sum())} games)")
print(forest_report.to_string())

# -- 5. what the averaging bought -------------------------------------------- #
# One row each: the configuration its own block carries forward, not the best
# row either of them printed.
summary = pd.concat(
    {"decision tree": tree_report.tail(1), "random forest": forest_report.tail(1)}
).droplevel(1)
print("\nthe model each block carries into 2026, on the same holdout")
print(summary.to_string())

forest = fit_model(FOREST, season.between(2011, HOLDOUT), FOREST_PARAMS, FOREST_FEATURES)
print(f"\nrefit on 2011-{HOLDOUT} ({int(season.between(2011, HOLDOUT).sum())} games) to predict 2026")

bayesian search (TPE), 40 trials over 12 folds (2013-2024)
params  : criterion=entropy  max_depth=17  min_samples_leaf=46  min_samples_split=167  max_features=0.46093878432770236  max_samples=0.5614658997297232
cv auc  : 0.6538   on all 10 features

permutation importance, out of fold: roc_auc lost when the column is shuffled
                           mean  seasons_up
away_pct_score_drive    0.04886          12
home_pct_score_drive    0.02581          12
home_pct_allowed_drive  0.01622          10
pct_home_win            0.01056          10
away_pct_allowed_drive  0.00592           9
pct_away_win            0.00363           9
day                     0.00003           4
playoff                 0.00000           0
week                   -0.00039           4
month                  -0.00060           4

kept    : away_pct_score_drive, home_pct_score_drive, home_pct_allowed_drive, pct_home_win, away_pct_allowed_drive, pct_away_win, day
dropped : playoff, week, month

retuned on the 7 surv

In [8]:
# The same 272 games and the same bracket the tree read, three hundred trees at
# a time. Where the two disagree is where the averaging changed somebody's season.
forest_standings = forecast_2026(forest, FOREST_FEATURES)

2026 regular season, 272 games predicted
W-L is the record the picks add up to, exp is the summed probabilities

AFC East    NE   14-3 (10.1)   BUF  13-4 ( 9.6)   MIA  5-12 ( 7.7)   NYJ  2-15 ( 6.3)
AFC North   PIT  10-7 ( 8.8)   BAL   8-9 ( 8.7)   CIN   8-9 ( 8.4)   CLE  2-15 ( 7.0)
AFC South   HOU  16-1 (10.4)   JAX  15-2 (10.3)   IND   9-8 ( 8.6)   TEN  1-16 ( 6.2)
AFC West    DEN  13-4 ( 9.6)   LAC   9-8 ( 8.8)   KC   7-10 ( 8.6)   LV   1-16 ( 5.8)
NFC East    PHI  10-7 ( 8.9)   DAL   9-8 ( 8.5)   NYG  5-12 ( 7.6)   WAS  3-14 ( 7.5)
NFC North   DET  13-4 ( 9.4)   CHI  11-6 ( 9.1)   MIN   9-8 ( 8.5)   GB    8-9 ( 8.5)
NFC South   TB   13-4 ( 9.1)   ATL   8-9 ( 8.4)   NO   4-13 ( 7.7)   CAR  2-15 ( 7.3)
NFC West    SEA  16-1 (10.8)   LAR  15-2 (10.0)   SF   11-6 ( 9.3)   ARI  2-15 ( 6.8)


2026 playoffs, predicted -- the better seed hosts every round

AFC seeds: 1.HOU  2.NE  3.DEN  4.PIT  5.JAX  6.BUF  7.LAC

AFC wild card   (HOU on a bye)
  (7) LAC at (2) NE    ->  NE   63.1%
  (6) 